# ANLP Project — Kaggle Session 2

**Scope:** LED fine-tuning + inference + final combined evaluation.

**Estimated time:** ~5–8 hours on T4.

**Before running:**
- Sidebar → Accelerator → **GPU T4 x2**
- Sidebar → Internet → **On**
- *(Optional but recommended)* Sidebar → **Add Data → Your Datasets** → attach the output of `marcopanciera/anlp-football-summarizer-session1` (Session 1's notebook output, after "Save Version"). The path will appear under `/kaggle/input/<dataset-name>/`. Cell 4 below will copy session 1's predictions into this run so the final metrics table is complete.

## 1. Clone repo

In [ ]:
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
WORK = '/kaggle/working'
REPO = f'{WORK}/ANLP_Project'
if not os.path.isdir(REPO):
    !git clone --quiet --branch setup/local-run https://github.com/christiandalfarra/ANLP_Project.git $REPO
%cd $REPO
!ls

## 2. Install dependencies

In [ ]:
!pip install -q sentencepiece bert_score rouge_score 'transformers>=4.40' 'accelerate>=0.30'

## 3. GPU check

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4. (Optional) Pull in Session 1 predictions

If you attached the Session 1 dataset, copy any prediction JSONs it contains into this session's `outputs/predictions/` so the final metrics table aggregates everything.

In [ ]:
import os, glob, shutil
os.makedirs('outputs/predictions', exist_ok=True)
found = glob.glob('/kaggle/input/*/ANLP_Project/outputs/predictions/*.json') + \
        glob.glob('/kaggle/input/*/outputs/predictions/*.json')
print(f'Found {len(found)} prediction files in attached datasets:')
for src in found:
    dst = os.path.join('outputs/predictions', os.path.basename(src))
    shutil.copy(src, dst)
    print(f'  copied {src}')
print()
print('Predictions in this session now:')
for p in sorted(glob.glob('outputs/predictions/*.json')):
    print(' ', p)

## 5. Fine-tune LED (~4–8 h)

Long-context fine-tuning with gradient checkpointing. Saves to `checkpoints/led/`.

In [ ]:
!python scripts/run_finetuning.py --model led --output_dir checkpoints/led

## 6. LED inference on 9 test matches (~20–30 min)

In [ ]:
!python scripts/run_inference_finetuned.py --model led --checkpoint checkpoints/led

## 7. Final evaluation across all conditions

In [ ]:
!python scripts/run_evaluation.py --device cuda

## 8. Inspect results

In [ ]:
import os, json, glob
for p in sorted(glob.glob('outputs/predictions/*.json')):
    print('===', os.path.basename(p))
    d = json.load(open(p))
    mid, summary = next(iter(d.items()))
    print(f'[{mid}]\n{summary[:600]}\n')
for p in sorted(glob.glob('outputs/results/*')):
    print(p)
    if p.endswith('.csv'):
        print(open(p).read())